# Making hero images with Sentinel-1 Normalised Radar Backscatter

This notebook makes a high quality image from Sentinel-1 Normalised Radar Backscatter (NRB) data, for an area and time of your choice.

The workflow is:
1. Search the DEA STAC API for a collection, area of interest and time range
2. Load the data at your chosen resolution
3. Optionally convert to ellipsoid sigma0 and mask layover and shadow, then apply a speckle filter, and combine the time steps into one image
4. Apply a style, which converts the data to decibels (dB) where the style uses it
5. Save a full resolution PNG

The styles and image saving come from [de_sar_demo/visualisation.py](../../de_sar_demo/visualisation.py), so you can also use them in your own notebooks.
Ready-made locations are defined in [locations.py](locations.py).

**Tip** - when trying a new area, start with a coarse resolution (e.g. 200 m) to check the area and style, then lower the resolution for the final image. Halving the pixel size needs four times the memory.

## Set up

In [ ]:
import os

# Use the environment's own PROJ/GDAL data, not any inherited paths
for _var in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(_var, None)

from dataclasses import replace
from pathlib import Path

from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
from odc.geo import BoundingBox
from odc.stac import load, configure_s3_access
from pystac_client import Client
import shapely
import xarray as xr

from locations import LOCATIONS, Location
from de_sar_demo.coverage import summarise_coverage_monthly
from de_sar_demo.masking import dilate_mask
from de_sar_demo.sar_utils import convert_gamma0_to_sigma0_ellipsoid_xr
from de_sar_demo.speckle_filters import lee_filter_xr
from de_sar_demo.visualisation import STYLES, apply_style, default_style_for_collection, save_image

stac_client = Client.open("https://explorer.dev.dea.ga.gov.au/stac")
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Settings

This is the only cell you need to edit. Choose a location from the table below, change part of one, or define your own.

| Location | Area | Collection | Time range | Resolution |
| --- | ----------- | --- | --- | --- |
| `pine_island_glacier_front` | The floating ice front of Pine Island Glacier, West Antarctica | HH | Jan 2025 | 60 m |
| `pine_island_glacier_front_zoom` | A closer view of the Pine Island Glacier ice front | HH | Jan 2025 | 20 m |
| `casey_station_vanderford_glacier` | Casey station and Vanderford Glacier | HH | 13 and 15 Jan 2022 (two passes) | 20 m |
| `antarctic_peninsula` | Crane Glacier and the Larsen B embayment | HH | Jan 2025 | 20 m |
| `erebus_ice_tongue_mcmurdo` | The Erebus Ice Tongue and McMurdo Station, Ross Island | HH | Jan to Feb 2026 | 20 m |
| `kangaroo_island_after_fire` | Kangaroo Island, shortly after the 2019-20 bushfires | VV+VH | 25 Jan to 1 Mar 2020 | 40 m |
| `kangaroo_island_before_fire` | Kangaroo Island, shortly before the 2019-20 bushfires | VV+VH | 1 Nov to 15 Dec 2019 | 40 m |
| `moree_farmland` | Cropping and irrigated farmland around Moree, New South Wales | VV+VH | Sep to Oct 2026 | 20 m |
| `lake_eyre` | Kati Thanda-Lake Eyre and its surrounding river channels | VV+VH | Sep to Oct 2026 | 60 m |
| `melbourne` | Melbourne and Port Phillip Bay | VV+VH | 20 Sep to 20 Oct 2026 | 20 m |

To add a location for everyone, add it to [locations.py](locations.py).

The other settings control how the image is made:

| Setting | Description |
| --- | ----------- |
| `STYLE` | A style from the table below, or `"auto"` to choose from the collection's polarisation |
| `PERCENTILES` | `None` for the style's fixed display ranges, or percentiles of the image (see below). Some locations recommend percentiles, as low backscatter areas can look dark with fixed ranges |
| `BACKSCATTER` | The backscatter convention to display: `"gamma0"` or `"sigma0_ellipsoid"`. Ellipsoid sigma0 removes the DEM-based brightness correction in gamma0, which can help where DEM errors show in the image, such as over ice |
| `CRS` | The output CRS, or `"auto"` for EPSG:3031 in Antarctica and EPSG:3577 elsewhere |
| `SPECKLE_SIZE` | The Lee filter window, in pixels. Larger is smoother, but loses fine detail |
| `COMPOSITE` | How to combine time steps: `"mean"`, `"median"` or `"first"`. Not used by `multitemporal_rgb` |
| `MASK_LAYOVER_SHADOW` | Whether to remove pixels affected by layover and shadow. These pixels become transparent holes in the image, mostly in steep terrain, so locations leave this off unless they set it. Without the mask, these pixels keep their distorted values, so they can look fine in an image but are not suited to analysis |
| `MASK_DILATION_METRES` | How far to grow the layover and shadow mask, to better cover ridge lines |
| `OUTPUT_DIR` | The folder for saved images |
| `CHECK_COVERAGE` | Whether to summarise the coverage of your area when looking for a good time period |
| `COVERAGE_START_MONTH`, `COVERAGE_END_MONTH` | The months to summarise when `CHECK_COVERAGE` is `True` |

In [ ]:
location = LOCATIONS["moree_farmland"]

# Change part of a location
# location = replace(LOCATIONS["melbourne"], resolution=20)

# Or define your own
# location = Location(
#     name="my_area",
#     description="My area of interest",
#     collection="ga_s1_nrb_iw_vv_vh_1",
#     aoi=BoundingBox(left=144.5, bottom=-38.5, right=145.5, top=-37.5, crs="EPSG:4326"),
#     start_date="2026-09-20",
#     end_date="2026-10-20",
#     resolution=200,
# )

STYLE = "auto"
PERCENTILES = location.percentiles  # or None, (2, 98), {"red": (2, 98), "blue": (5, 95)}
BACKSCATTER = location.backscatter  # or "gamma0", "sigma0_ellipsoid"
CRS = "auto"
SPECKLE_SIZE = 5
COMPOSITE = "mean" if location.backscatter == "gamma0" else "median"
MASK_LAYOVER_SHADOW = location.mask_layover_shadow  # or True, False
MASK_DILATION_METRES = 60
OUTPUT_DIR = Path("outputs")
CHECK_COVERAGE = False
COVERAGE_START_MONTH = "2025-01"
COVERAGE_END_MONTH = "2026-10"

### Available styles

| Style | Needs | Description |
| --- | --- | ----------- |
| `ice_blue` | HH | White-to-blue scale for ice, in dB. Default for HH collections |
| `vv_vh_false_colour_linear` | VV+VH | Red = VV, green = VH, blue = VH/VV, in linear scale. Default for VV+VH collections |
| `dual_pol_db` | VV+VH or HH+HV | Red = co-pol, green = cross-pol, blue = co-pol minus cross-pol, in dB. Default for HH+HV collections |
| `greyscale_db` | Any | Greyscale of a single band in dB, using the co-polarised band if there are two. Default for VV collections |
| `multitemporal_rgb` | Any, with at least 3 time steps | The early, middle and late parts of the time range as red, green and blue, in dB. Stable areas look grey, and changes show as colour. Only shows pixels with data in all three parts, so check that the passes in your time range cover the area |

### Display ranges

By default, each style uses fixed display ranges, so colours can be compared between images.
Setting `PERCENTILES` stretches the display range to suit each image instead, which often gives more contrast, but means colours are no longer comparable between images or dates.
Percentiles are calculated on the values each channel displays, so they're in dB for dB styles, and linear for `vv_vh_false_colour_linear` (with the VH/VV ratio for blue).

The ranges used are printed when the image is made, so you can see what a percentile stretch chose.

## Find a good time period

To explore what each Collection 1 product holds, open it in the DEA Explorer:

| Polarisation | Explorer |
| --- | --- |
| HH | [ga_s1_nrb_iw_hh_1](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_hh_1) |
| HH+HV | [ga_s1_nrb_iw_hh_hv_1](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_hh_hv_1) |
| VV | [ga_s1_nrb_iw_vv_1](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_vv_1) |
| VV+VH | [ga_s1_nrb_iw_vv_vh_1](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_vv_vh_1) |

When `CHECK_COVERAGE` is `True`, the next cell summarises how well your chosen collection covers your area in each month, between `COVERAGE_START_MONTH` and `COVERAGE_END_MONTH`.
It compares item footprints with the area without loading any data, so it's quick, but footprints can slightly overstate the pixels with data.

| Column | Description |
| --- | ----------- |
| `passes` | The number of satellite passes that touch the area |
| `coverage_percent` | The percentage of the area covered by at least one pass. Aim for 100 to avoid gaps in the image |
| `min_passes` | The fewest passes over any point checked in the area, as a guide. A value of 0 means there's a gap, but footprints can overstate this, so the image may have fewer passes in places |
| `median_passes` | The number of passes over a typical point. More passes give a smoother composite |
| `best_pass_percent` | The percentage of the area covered by the most complete single pass. Aim for 100 if you want a single-date image |

Once you've chosen a time period, set `start_date` and `end_date` in the settings, and re-run from there.
Coverage doesn't tell you how the image will look, as conditions like sea ice change through the year, so try a coarse resolution first.

In [ ]:
if CHECK_COVERAGE:
    display(
        summarise_coverage_monthly(
            stac_client,
            location.collection,
            location.aoi,
            COVERAGE_START_MONTH,
            COVERAGE_END_MONTH,
        )
    )

## Search for data

In [ ]:
if BACKSCATTER not in ("gamma0", "sigma0_ellipsoid"):
    raise ValueError(f"BACKSCATTER must be one of 'gamma0' or 'sigma0_ellipsoid', got: {BACKSCATTER}")

style_name = default_style_for_collection(location.collection) if STYLE == "auto" else STYLE
print(location.description)
print(f"Style: {style_name} - {STYLES[style_name].title}")
print(f"Backscatter: {BACKSCATTER}")

aoi_shape = shapely.geometry.shape(location.aoi.polygon)
items = stac_client.search(
    collections=[location.collection],
    datetime=f"{location.start_date}/{location.end_date}",
    intersects=aoi_shape,
).item_collection()

print(f"Found {len(items)} items")
if len(items) == 0:
    raise ValueError("No items found. Check the coverage of this collection, or try a different area or time range.")

## Load the data

The backscatter bands depend on the collection's polarisation, so they are read from the first item.
Converting to ellipsoid sigma0 also needs the `oa_gamma0_to_beta0_ratio` and `oa_incidence_angle` layers.

Over Antarctica, we group by absolute orbit to combine the bursts from each satellite pass.
Elsewhere, we group by solar day.

In [ ]:
is_antarctic = location.aoi.top < -60
output_crs = ("EPSG:3031" if is_antarctic else "EPSG:3577") if CRS == "auto" else CRS
group_by = "sat:absolute_orbit" if is_antarctic else "solar_day"

gamma0_assets = [name for name in items[0].assets if name.endswith("_gamma0")]
mask_asset = "oa_layover_shadow_mask"
ratio_asset = "oa_gamma0_to_beta0_ratio"
angle_asset = "oa_incidence_angle"
use_sigma0 = BACKSCATTER == "sigma0_ellipsoid"

ds = load(
    items,
    bands=gamma0_assets + ([mask_asset] if MASK_LAYOVER_SHADOW else []) + ([ratio_asset, angle_asset] if use_sigma0 else []),
    geopolygon=aoi_shape,
    crs=output_crs,
    resolution=location.resolution,
    groupby=group_by,
    chunks={},
).sortby("time")

gigabytes = sum(ds[band].size * ds[band].dtype.itemsize for band in ds.data_vars) / 1e9
print(f"Image size: {ds.sizes['x']} x {ds.sizes['y']} pixels, {ds.sizes['time']} time steps")
print(f"Memory needed to load: about {gigabytes:.2f} GB")
if gigabytes > 8:
    print("Warning: this is a large load. Consider a coarser resolution, or a smaller area or time range.")

In [ ]:
with ProgressBar():
    ds = ds.compute()

## Convert, mask, filter and combine

The order of these steps matters:
1. **Convert to ellipsoid sigma0**, if selected, while the data is still linear and unfiltered.
2. **Mask layover and shadow** before filtering, so that affected pixels don't spread into their neighbours. The mask is dilated to better cover ridge lines, by a set distance on the ground so that it doesn't grow at coarse resolutions.
3. **Apply the Lee speckle filter** to each time step in linear scale.
4. **Combine time steps** in linear scale, because averaging dB values would bias the result towards low backscatter.

Conversion to dB happens in the next section, as part of the styles that use it.
`vv_vh_false_colour_linear` has display ranges defined in linear scale, so it doesn't use dB.

In [ ]:
if use_sigma0:
    backscatter = xr.Dataset(
        {
            name.replace("gamma0", "sigma0_ellipsoid"): convert_gamma0_to_sigma0_ellipsoid_xr(
                ds[name], ds[ratio_asset], ds[angle_asset]
            )
            for name in gamma0_assets
        }
    )
else:
    backscatter = ds[gamma0_assets]

if MASK_LAYOVER_SHADOW:
    # 1 = shadow, 2 = layover, 3 = shadow and layover
    dilation_radius = round(MASK_DILATION_METRES / location.resolution)
    layover_shadow = dilate_mask(ds[mask_asset].isin([1, 2, 3]), dilation_radius=dilation_radius)
    backscatter = backscatter.where(~layover_shadow)

filtered = backscatter.map(lee_filter_xr, size=SPECKLE_SIZE)

if STYLES[style_name].uses_time:
    to_style = filtered
elif COMPOSITE == "mean":
    to_style = filtered.mean("time")
elif COMPOSITE == "median":
    to_style = filtered.median("time")
elif COMPOSITE == "first":
    to_style = filtered.isel(time=0, drop=True)
else:
    raise ValueError(f"COMPOSITE must be one of 'mean', 'median' or 'first', got: {COMPOSITE}")

## Make the image

In [ ]:
rgba = apply_style(to_style, style_name, percentiles=PERCENTILES)

print("Display ranges used:")
for channel, (low, high) in rgba.attrs["scale_ranges"].items():
    print(f"    {channel}: {low:.5g} to {high:.5g}")

coverage = 100 * float(rgba.sel(band="alpha").mean())
print(f"Pixels with data: {coverage:.1f}%")

### Save the full resolution image

In [ ]:
file_name = f"{location.name}_{BACKSCATTER}_{style_name}_{location.start_date}_{location.end_date}_{location.resolution}m.png"
output_path = save_image(rgba, OUTPUT_DIR / file_name)
print(f"Saved {rgba.sizes['x']} x {rgba.sizes['y']} pixel image to {output_path}")

### Preview

This preview is reduced in size to keep the notebook small. Open the saved PNG to see the full resolution image.

In [ ]:
step = max(1, max(rgba.sizes["x"], rgba.sizes["y"]) // 1500)
preview = rgba.isel(x=slice(None, None, step), y=slice(None, None, step))

fig, ax = plt.subplots(figsize=(10, 10 * preview.sizes["y"] / preview.sizes["x"]))
preview.plot.imshow(ax=ax, rgb="band", add_labels=False)
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()